In [1]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import random
import numpy as np
import torchvision.transforms.functional as FT


class LEVIRCDAugmentation:
    """
    LEVIR-CD 데이터셋(img_a, img_b, mask) 구조에 맞게 구현.
    img_a, img_b : PIL.Image (RGB)
    mask         : PIL.Image (L)
    Augmentation method : 논문 github 
    """
    def __init__(self, img_size: int, crop_area: int | None = None):
        self.img_size  = img_size
        # RandomCropResize의 crop_area: 원본에서 int(7./224.*width) 공식 사용
        self.crop_area = crop_area if crop_area is not None else int(7. / 224. * img_size)

    # ------------------------------------------------------------------ #
    # 내부 헬퍼                                                            #
    # ------------------------------------------------------------------ #
    @staticmethod
    def _to_numpy(img: Image.Image) -> np.ndarray:
        return np.array(img)

    @staticmethod
    def _to_pil(arr: np.ndarray, mode: str) -> Image.Image:
        return Image.fromarray(arr, mode=mode)

    # Scale : 다른 크기 입력 이미지일때 
    def scale( self, img_a: Image.Image, img_b: Image.Image, mask: Image.Image, ):
        """고정 크기로 리사이즈. RGB → BILINEAR, mask → NEAREST."""
        size = (self.img_size, self.img_size)
        img_a = img_a.resize(size, Image.BILINEAR)
        img_b = img_b.resize(size, Image.BILINEAR)
        mask  = mask.resize(size,  Image.NEAREST)
        return img_a, img_b, mask

    # RandomCropResize             
    def random_crop_resize(self, img_a: Image.Image, img_b: Image.Image, mask: Image.Image,):
        """
        * 50% 확률로 이미지 가장자리를 crop_area만큼 잘라낸 뒤원래 크기로 복원.
          세 이미지에 동일한 crop 적용.
        의미:
        1. 가장자리 물체나 노이즈에 대한 강건성(robustness) 향상
        2. 이미지를 약간 확대한 것과 유사한 효과 (zoom-in augmentation)
        3. crop 비율이 3.1%로 작아서 변화 탐지에 중요한 경계 정보 손실을 최소화
        """
        if random.random() >= 0.5:
            return img_a, img_b, mask

        w, h = img_a.size          # PIL: (width, height)
        x1 = random.randint(0, self.crop_area)
        y1 = random.randint(0, self.crop_area)

        # (left, upper, right, lower)
        box = (x1, y1, w - x1, h - y1)

        img_a = img_a.crop(box).resize((w, h), Image.BILINEAR)
        img_b = img_b.crop(box).resize((w, h), Image.BILINEAR)
        mask  = mask.crop(box).resize((w, h),  Image.NEAREST)
        return img_a, img_b, mask

    # RandomFlip  # horizontal or vertical flip
    def random_flip(self, img_a: Image.Image, img_b: Image.Image, mask: Image.Image,):
        """
        1/2 확률 수평(horizontal) 플립, 1/2 확률 수직(vertical) 플립.
        원본 코드: cv2.flip(img, 0) → 수평, cv2.flip(img, 1) → 수직
        """
        if random.random() < 0.5:         
            img_a = transforms.functional.hflip(img_a)
            img_b = transforms.functional.hflip(img_b)
            mask  = transforms.functional.hflip(mask)
        if random.random() < 0.5:          
            img_a = transforms.functional.vflip(img_a)
            img_b = transforms.functional.vflip(img_b)
            mask  = transforms.functional.vflip(mask)
        return img_a, img_b, mask

    # RandomExchange 
    def random_exchange(self, img_a: Image.Image, img_b: Image.Image, mask: Image.Image,):
        """
        1/2 확률 pre/post 이미지(A ↔ B)를 교환.
        변화탐지에서 시간 순서 불변성을 학습.
        """
        if random.random() < 0.5:
            img_a, img_b = img_b, img_a
        return img_a, img_b, mask

    # 전체 파이프라인                                                       #
    def __call__(self, img_a: Image.Image, img_b: Image.Image, mask: Image.Image,):
        img_a, img_b, mask = self.scale(img_a, img_b, mask)
        img_a, img_b, mask = self.random_crop_resize(img_a, img_b, mask)
        img_a, img_b, mask = self.random_flip(img_a, img_b, mask)
        img_a, img_b, mask = self.random_exchange(img_a, img_b, mask)
        return img_a, img_b, mask


class LEVIRCDDataset(Dataset):
    def __init__(self, root, split="train", img_size=256, augment=False):
        self.img_size = img_size
        self.augment  = augment

        self.root  = Path(root)
        self.split = split

        list_file = self.root / "list" / f"{split}.txt"
        with open(list_file, "r") as f:
            self.filenames = [line.strip() for line in f if line.strip()]

        self.dir_a     = self.root / split / "A"
        self.dir_b     = self.root / split / "B"
        self.dir_label = self.root / split / "label"

        # augmentation (train only)
        self.augmentation = LEVIRCDAugmentation(img_size=img_size) if augment else None

        # Scale만 적용하는 val용 transform
        self.val_resize = transforms.Resize((img_size, img_size))
        self.val_resize_nearest = transforms.Resize(
            (img_size, img_size),
            interpolation=transforms.InterpolationMode.NEAREST,
        )

        self.to_tensor   = transforms.ToTensor()
        self.normalize   = transforms.Normalize( mean=[0.485, 0.456, 0.406],
                                                 std =[0.229, 0.224, 0.225], )

    def _img_to_tensor(self, img: Image.Image) -> torch.Tensor:
        return self.normalize(self.to_tensor(img))

    def _mask_to_tensor(self, mask: Image.Image) -> torch.Tensor:
        t = self.to_tensor(mask)          # [1, H, W], 0~1
        return (t > 0.5).long().squeeze(0)  # [H, W]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx: int) -> dict:
        fname = self.filenames[idx]
        img_a = Image.open(self.dir_a     / fname).convert("RGB")
        img_b = Image.open(self.dir_b     / fname).convert("RGB")
        mask  = Image.open(self.dir_label / fname).convert("L")

        if self.augment and self.augmentation is not None:
            # Scale + RandomCropResize + RandomFlip + RandomExchange
            img_a, img_b, mask = self.augmentation(img_a, img_b, mask)
        else:
            # val: Scale만
            img_a = self.val_resize(img_a)
            img_b = self.val_resize(img_b)
            mask  = self.val_resize_nearest(mask)

        img_a_t = self._img_to_tensor(img_a)
        img_b_t = self._img_to_tensor(img_b)
        image   = torch.stack([img_a_t, img_b_t], dim=0)  # [2, 3, H, W]
        mask_t  = self._mask_to_tensor(mask)              # [H, W]

        return {"image": image, "mask": mask_t, "filename": fname}

def get_dataloader(root, split, img_size=256, batch_size=4, num_workers=0, shuffle=None):
    is_train = split == "train"
    dataset = LEVIRCDDataset(root=root, split=split, img_size=img_size, augment=is_train)
    return DataLoader(dataset,
                      batch_size=batch_size,
                      shuffle=is_train if shuffle is None else shuffle,
                      num_workers=num_workers,
                      pin_memory=True,
                      drop_last=is_train)


In [2]:
import shutil
from datetime import datetime
import torch.nn.functional as F
import lightning as L
from lightning.pytorch.callbacks import (ModelCheckpoint, EarlyStopping, 
                                         LearningRateMonitor, Callback
)
from lightning.pytorch.loggers import TensorBoardLogger
from torchgeo.models import ChangeViT
from torchgeo.models.changevit import DetailCaptureModule

# ─────────────────────────────────────────────
# 1. Config
"""
steps_per_epoch = len(train_loader)           # 445
MAX_ITERS       = steps_per_epoch * CFG["max_epochs"]  # 17,800

steps_per_epoch = 7120 / 16 = 445 steps
total_iters     = 445 x 40  = 17,800 iters -> 이 상태에서는 poly schedule의 LR이 거의 줄어들지 않음

마지막 iter의 lr = 2e-4 x (1 - 17800/80000)^0.9
                 = 2e-4 x 0.777^0.9
                 ≈ 2e-4 x 0.802
                 ≈ 1.6e-4   ← 거의 안 줄어듦
"""

CFG = dict(
    data_root    = "D:/Project-Dataset/LEVIR-CD-256",
    seed         = 16,
    img_size     = 256,
    batch_size   = 16,
    num_workers  = 0,
    max_epochs   = 180,
    lr           = 2e-4,
    weight_decay = 1e-4,
    backbone     = 'vit_small_patch16_dinov3.lvd1689m',   # timm 
    # backbone     = "vit_small_patch14_dinov2.lvd142m",  # timm model
    # backbone     = 'facebook/dinov3-vits16-pretrain-lvd1689m',  # hugging face
    log_dir      = "logs/changevit",
    ckpt_base    = "checkpoints/changevit",   # 실험별 서브폴더 여기 아래 생성됨
)

def init_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

init_seed(CFG["seed"])
# ─────────────────────────────────────────────
# 실험 ID: 훈련 시작 시각 → 항상 새 폴더 생성
RUN_ID   = datetime.now().strftime("%Y%m%d_%H%M%S")
CKPT_DIR = Path(CFG["ckpt_base"]) / RUN_ID
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────
# 2. Best-checkpoint → best.pth 복사 콜백
class SaveBestPth(Callback):
    """
    ModelCheckpoint가 best 파일을 갱신할 때마다
    같은 폴더에 best.pth 로 복사해 둔다.
    """
    def __init__(self, ckpt_callback: ModelCheckpoint):
        self.ckpt_callback = ckpt_callback

    def on_validation_end(self, trainer, pl_module):
        best = self.ckpt_callback.best_model_path
        if best and Path(best).exists():
            dst = Path(best).parent / "best.pth"
            shutil.copy2(best, dst)


# ─────────────────────────────────────────────
# 3. LightningModel
class LightningModel(L.LightningModule):
    def __init__(self, cfg: dict, max_iters: int):
        super().__init__()
        self.save_hyperparameters({**cfg, "max_iters": max_iters})
        self.max_iters  = max_iters
        self._optimizer = None          # setup()에서 단 한 번 생성

        self.model = ChangeViT(
            backbone    = cfg["backbone"],
            img_size    = cfg["img_size"],
            in_channels = 3,
            num_classes = 1,
            pretrained  = True,
        )
        # detailcapture backbone 변경 
        new_detail = DetailCaptureModule(
            in_channels = 6,
            backbone    = "resnet34.a1_in1k",  
            pretrained  = True,
        )
        self.model.detail_capture = new_detail
        
    # ── LR schedule (논문 poly + warm-up) ────────────
    def adjust_learning_rate(self, cur_iter: int) -> float:
        """
        논문 수식: lr = base_lr x (1 - cur_iter / max_iter)^0.9
        epoch=0 초기 200 iter: linear warm-up
            lr = base_lr x [0.9 x (iter+1)/200 + 0.1]
        cur_iter: 전체 step 기준 (epoch x steps_per_epoch + batch_idx)
        """
        base_lr = self.hparams["lr"]
        # warm-up 구간 (첫 번째 epoch 초반 200 iter)
        if cur_iter < 200:
            lr = base_lr * (0.9 * (cur_iter + 1) / 200 + 0.1)
        else:
            lr = base_lr * (1 - cur_iter / self.max_iters) ** 0.9
        # eta_min 하한 (논문에는 명시 없으나 안전장치)
        lr = max(lr, 1e-6)
        for pg in self._optimizer.param_groups:
            pg["lr"] = lr
        return lr

    # ── Loss ─────────────────────────────────────────
    def BCEDiceLoss(self, logits, targets):
        targets_f = targets.unsqueeze(1).float()
        bce   = F.binary_cross_entropy_with_logits(logits, targets_f)
        probs = torch.sigmoid(logits.float())
        inter = (probs * targets_f).sum()
        eps   = 1e-5
        dice  = (2 * inter + eps) / (probs.sum() + targets_f.sum() + eps)
        return bce + 1 - dice

    # ── Metrics: F1 / IoU / OA ───────────────────────
    @torch.no_grad()
    def _metrics(self, logits, targets):
        """
        Returns: f1, iou, oa  (pixel-level binary)
        OA = Overall Accuracy = (TP + TN) / N
        """
        preds = (torch.sigmoid(logits) > 0.5).long().squeeze(1)   # [B,H,W]
        tp = ((preds == 1) & (targets == 1)).sum().float()
        tn = ((preds == 0) & (targets == 0)).sum().float()
        fp = ((preds == 1) & (targets == 0)).sum().float()
        fn = ((preds == 0) & (targets == 1)).sum().float()

        precision = tp / (tp + fp + 1e-5)
        recall    = tp / (tp + fn + 1e-5)
        f1  = 2 * precision * recall / (precision + recall + 1e-5)
        iou = tp / (tp + fp + fn + 1e-5)
        oa  = (tp + tn) / (tp + tn + fp + fn + 1e-5)
        return f1, iou, oa

    # ── Gradient norm ─────────────────────────────────
    def log_gradient_norms(self):
        total = torch.tensor(0.0, device=self.device)
        for p in self.parameters():
            if p.grad is not None:
                total += p.grad.detach().norm(2) ** 2
        return total.sqrt()

    # def on_train_epoch_end(self, trainer, pl_module):
    #     if self.trainer.current_epoch == 0:
    #         self.trainer.save_checkpoint("first_epoch.ckpt")


    # ── Steps ─────────────────────────────────────────
    def training_step(self, batch, batch_idx):
        # poly LR 갱신
        steps_per_epoch = self.trainer.num_training_batches
        cur_iter = self.current_epoch * steps_per_epoch + batch_idx
        lr = self.adjust_learning_rate(cur_iter)

        image, mask = batch["image"], batch["mask"]
        logits = self.model(image)
        loss   = self.BCEDiceLoss(logits, mask)

        f1, iou, oa = self._metrics(logits, mask)
        self.log("train/loss", loss, prog_bar=True,  on_step=True,  on_epoch=True)
        self.log("train/f1",   f1,   prog_bar=False, on_step=False, on_epoch=True)
        self.log("train/iou",  iou,  prog_bar=False, on_step=False, on_epoch=True)
        self.log("train/oa",   oa,   prog_bar=False, on_step=False, on_epoch=True)
        self.log("train/lr",   lr,   prog_bar=True,  on_step=True,  on_epoch=False)
        return loss

    def on_after_backward(self):
        gnorm = self.log_gradient_norms()
        # print(f"[hook 호출됨] grad_norm={gnorm:.4f}")  # 확인
        self.log("train/grad_norm", gnorm, prog_bar=False, on_step=True, on_epoch=False)

    def validation_step(self, batch, batch_idx):
        image, mask = batch["image"], batch["mask"]
        logits = self.model(image)
        loss   = self.BCEDiceLoss(logits, mask)

        f1, iou, oa = self._metrics(logits, mask)
        self.log("val/loss", loss, prog_bar=True,  on_step=False, on_epoch=True)
        self.log("val/f1",   f1,  prog_bar=True,  on_step=False, on_epoch=True)
        self.log("val/iou",  iou, prog_bar=True,  on_step=False, on_epoch=True)
        self.log("val/oa",   oa,  prog_bar=True,  on_step=False, on_epoch=True)

    # ── Optimizer (스케줄러 없음 — 수동 poly 사용) ────
    def configure_optimizers(self):
        self._optimizer = torch.optim.AdamW(
            self.parameters(),
            lr           = self.hparams["lr"],
            betas        = (0.9, 0.99),          # 논문 설정
            weight_decay = self.hparams["weight_decay"],
            eps=1e-08  # github ChangeViT/main.py
        )
        return self._optimizer   # lr_scheduler 반환 안 함

In [3]:
# ─────────────────────────────────────────────
# 4. DataLoaders
train_loader = get_dataloader( root=CFG["data_root"], split="train",
    img_size=CFG["img_size"], batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
)
val_loader = get_dataloader( root=CFG["data_root"], split="val",
    img_size=CFG["img_size"], batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
)

# max_iters 계산 (논문: LEVIR-CD 80K) => Optimizer Control
MAX_ITERS = 80000

# ─────────────────────────────────────────────
# 5. Callbacks & Logger
ckpt_cb = ModelCheckpoint(
    dirpath   = str(CKPT_DIR),
    filename  = "epoch{epoch:02d}-val_f1{val/f1:.4f}",
    monitor   = "val/f1",
    mode      = "max",
    save_top_k= 3,
    auto_insert_metric_name=False,
)

callbacks = [
    ckpt_cb,
    SaveBestPth(ckpt_cb),          # best.pth 자동 복사
    # val-f1 값 추적: val-f1의 min_delta 값 0.01이 5 epochs 동안 유지되면 stop training
    # val-f1 = 0.71 => 71%   
    # Max epochs까지 훈련하려면 comment EarlyStopping out 
    # EarlyStopping(monitor="val/f1", patience=20, mode="max", min_delta=0.001), 
    LearningRateMonitor(logging_interval="step"),
]

logger = TensorBoardLogger(
    save_dir = CFG["log_dir"],
    name     = RUN_ID,              # 실험별 로그 폴더도 분리
)

# ─────────────────────────────────────────────
# 6. Train
model = LightningModel(CFG, max_iters=MAX_ITERS)

trainer = L.Trainer(
    # max_epochs = -1,          # epoch 제한 해제
    max_epochs        = CFG["max_epochs"],
    accelerator       = "gpu",
    devices           = 1,
    precision         = "16-mixed",
    callbacks         = callbacks,
    logger            = logger,
    log_every_n_steps = 10,
    # deterministic     = True,   # ← init_seed만으로는 Lightning의 내부 난수까지 완전히 제어되지 않으므로,  Trainer에 deterministic 옵션을 함께 추가하면 완전한 재현성을 확보
)
# tensorboard --logdir=path/to/logs
trainer.fit(model, train_loader, val_loader)

print(f"\n체크포인트 저장 위치: {CKPT_DIR}")
print(f"Best model: {ckpt_cb.best_model_path}")

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 3060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
d:\Project-TorchGeo\.venv_py312\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 3

┏━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ChangeViT │ 26.2 M │ train │     0 │
└───┴───────┴───────────┴────────┴───────┴───────┘

Trainable params: 26.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.2 M                                                                                               
Total estimated model params size (MB): 104                                                                        
Modules in train mode: 361                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

d:\Project-TorchGeo\.venv_py312\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

d:\Project-TorchGeo\.venv_py312\Lib\site-packages\lightning\pytorch\utilities\data.py:79: Trying to infer the 
`batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use 
`self.log(..., batch_size=batch_size)`.

d:\Project-TorchGeo\.venv_py312\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

d:\Project-TorchGeo\.venv_py312\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
